Attempting to segment cells using nuclear tracking and Voronoi diagrams

In [ ]:
import os
import sys
import zarr
import pandas as pd
import numpy as np
from scipy import ndimage
from scipy.spatial import Voronoi, voronoi_plot_2d
from skimage import filters, morphology, measure, segmentation
from skimage.feature import peak_local_max
import trackpy as tp
import napari
from typing import Tuple, List, Dict
from tqdm import tqdm

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

In [ ]:
# Load the zarr file
base_dir = r'Z:\Abhi\LLSM_Analysis'
zarr_file_directory = 'controlOS_analysis/zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_file_directory)

zarr_file = zarr.open(zarr_full_path, mode='r')

In [ ]:
class NucleiTracker3D:
    """
    3D nuclei segmentation and tracking pipeline for hiPSC timelapse data.
    Adapted from 2D amnioserosa analysis to handle 3D volumetric data.
    """
    
    def __init__(self, sigma: float = 2.0, min_distance: int = 10, 
                 threshold_abs: float = None, threshold_rel: float = 0.1):
        """
        Initialize tracker with parameters.
        
        Parameters:
        -----------
        sigma : float
            Standard deviation for Gaussian filter (in pixels)
        min_distance : int
            Minimum distance between detected peaks (nuclei centers)
        threshold_abs : float
            Absolute threshold for peak detection
        threshold_rel : float
            Relative threshold for peak detection (fraction of max)
        """
        self.sigma = sigma
        self.min_distance = min_distance
        self.threshold_abs = threshold_abs
        self.threshold_rel = threshold_rel
        self.tracks = None
        self.segmentation_masks = []


    def detect_nuclei_centers(self, volume: np.ndarray) -> np.ndarray:
        """
        Detect nuclei centers in 3D volume using local maxima detection.
        
        Parameters:
        -----------
        volume : np.ndarray
            3D image volume (z, y, x)
            
        Returns:
        --------
        centers : np.ndarray
            Array of nuclei center coordinates (n_nuclei, 3)
        """
        # Apply 3D Gaussian filter
        filtered = filters.gaussian(volume, sigma=self.sigma)
        
        # Invert image (assuming nuclei are bright)
        # For fluorescent nuclei, you might not need this step
        inverted = filtered.max() - filtered
        
        # Find local maxima in 3D
        local_maxima = peak_local_max(
            inverted,
            min_distance=self.min_distance,
            threshold_abs=self.threshold_abs,
            threshold_rel=self.threshold_rel,
            exclude_border=True
        )

        return local_maxima
    
    def segment_volume_watershed(self, volume: np.ndarray, 
                                 centers: np.ndarray) -> np.ndarray:
        """
        Segment nuclei using marker-controlled watershed in 3D.
        
        Parameters:
        -----------
        volume : np.ndarray
            3D image volume
        centers : np.ndarray
            Nuclei center coordinates
            
        Returns:
        --------
        labels : np.ndarray
            Labeled segmentation mask
        """
        # Create markers from centers
        markers = np.zeros(volume.shape, dtype=np.int32)
        for i, (z, y, x) in enumerate(centers):
            markers[z, y, x] = i + 1
        
        # Dilate markers slightly
        markers = morphology.dilation(markers, morphology.ball(2))
        
        # Apply Gaussian filter for smoother gradients
        filtered = filters.gaussian(volume, sigma=self.sigma/2)
        
        # Compute gradient magnitude for watershed
        gradient = filters.sobel(filtered)
        
        # Perform watershed
        labels = segmentation.watershed(gradient, markers, mask=volume > 0)
        
        return labels
    
    def create_voronoi_segmentation(self, volume_shape: Tuple[int, int, int],
                                    centers: np.ndarray) -> np.ndarray:
        """
        Create 3D Voronoi tessellation for cell boundary estimation.
        Note: scipy.spatial.Voronoi works in 3D but can be memory intensive.
        
        Parameters:
        -----------
        volume_shape : tuple
            Shape of the volume (z, y, x)
        centers : np.ndarray
            Nuclei center coordinates
            
        Returns:
        --------
        labels : np.ndarray
            Voronoi-based segmentation
        """
        # Create coordinate grid
        z, y, x = np.indices(volume_shape)
        coords = np.column_stack((z.ravel(), y.ravel(), x.ravel()))
        
        # For each voxel, find nearest nucleus center
        from scipy.spatial import cKDTree
        tree = cKDTree(centers)
        _, labels = tree.query(coords)
        
        # Reshape to volume
        labels = labels.reshape(volume_shape) + 1  # +1 to avoid 0 label
        
        return labels


    def track_nuclei_across_time(self, time_series: np.ndarray,
                                    search_range: int = 15,
                                    memory: int = 3) -> pd.DataFrame:
        """
        Track nuclei across time using trackpy.
        
        Parameters:
        -----------
        time_series : np.ndarray
            4D array (t, z, y, x)
        search_range : int
            Maximum displacement between frames
        memory : int
            Number of frames a particle can disappear
            
        Returns:
        --------
        tracks : pd.DataFrame
            Tracked particle trajectories
        """
        all_features = []
        
        for t in tqdm(range(len(time_series)), desc="Detecting nuclei centers"):
            volume = time_series[t]
            centers = self.detect_nuclei_centers(volume)
            
            # Create DataFrame for trackpy
            features = pd.DataFrame(
                centers, 
                columns=['z', 'y', 'x']
            )
            features['frame'] = t
            features['mass'] = 1  # Placeholder, could use actual intensity
            all_features.append(features)
        
        # Combine all frames
        features_df = pd.concat(all_features, ignore_index=True)
        
        # Link trajectories using trackpy
        self.tracks = tp.link(
            features_df,
            search_range=search_range,
            memory=memory,
            pos_columns=['z', 'y', 'x']
        )
        
        return self.tracks

    def process_timelapse(self, time_series: np.ndarray,
                            method: str = 'watershed') -> Tuple[List[np.ndarray], pd.DataFrame]:
        """
        Complete pipeline for 3D timelapse processing.
        
        Parameters:
        -----------
        time_series : np.ndarray
            4D array (t, z, y, x)
        method : str
            Segmentation method ('watershed' or 'voronoi')
            
        Returns:
        --------
        segmentations : list
            List of segmented volumes for each timepoint
        tracks : pd.DataFrame
            Tracked nuclei trajectories
        """
        segmentations = []
        
        # First pass: detect and track centers
        tracks = self.track_nuclei_across_time(time_series)
        
        # Second pass: segment each timepoint
        for t, volume in enumerate(tqdm(time_series, desc="Segmenting volumes")):
            # Get centers for this timepoint
            frame_tracks = tracks[tracks['frame'] == t]
            centers = frame_tracks[['z', 'y', 'x']].values.astype(int)
            
            # Segment based on method
            if method == 'watershed':
                labels = self.segment_volume_watershed(volume, centers)
            elif method == 'voronoi':
                labels = self.create_voronoi_segmentation(volume.shape, centers)
            else:
                raise ValueError(f"Unknown method: {method}")
            
            segmentations.append(labels)
        
        self.segmentation_masks = segmentations
        return segmentations, tracks


    def compute_cell_features(self, volume: np.ndarray, 
                                labels: np.ndarray) -> pd.DataFrame:
        """
        Extract morphological features from segmented cells.
        
        Parameters:
        -----------
        volume : np.ndarray
            Original image volume
        labels : np.ndarray
            Segmentation labels
            
        Returns:
        --------
        features : pd.DataFrame
            Cell features including volume, intensity, shape
        """
        props = measure.regionprops_table(
            labels, 
            volume,
            properties=[
                'label', 'centroid', 'area', 'mean_intensity',
                'major_axis_length', 'minor_axis_length',
                'extent', 'solidity'
            ]
        )
        
        features = pd.DataFrame(props)
        features.rename(columns={
            'centroid-0': 'z',
            'centroid-1': 'y', 
            'centroid-2': 'x',
            'area': 'volume'
        }, inplace=True)
        
        return features


    def visualize_with_napari(self, time_series: np.ndarray,
                                tracks: pd.DataFrame = None):
        """
        Visualize results using napari.
        
        Parameters:
        -----------
        time_series : np.ndarray
            4D array (t, z, y, x)
        tracks : pd.DataFrame
            Tracked trajectories
        """
        viewer = napari.Viewer()
        
        # Add raw data
        viewer.add_image(time_series, name='Raw data')
        
        # Add segmentation if available
        if self.segmentation_masks:
            viewer.add_labels(
                np.array(self.segmentation_masks),
                name='Segmentation'
            )
        
        # Add tracks if available
        if tracks is not None:
            # Format tracks for napari
            track_data = tracks[['particle', 'frame', 'z', 'y', 'x']].values
            viewer.add_tracks(track_data, name='Tracks')
        
        return viewer

In [ ]:
def example_usage():
    """
    Example of how to use the tracker with hiPSC data.
    """
    # Load your 4D data (t, z, y, x)
    # data = load_your_data()  # Replace with actual data loading
    
    # For demonstration, create synthetic data
    import numpy as np
    data = zarr_file[0:2, 2, :, 100:400, 50:200]
    
    # Initialize tracker
    tracker = NucleiTracker3D(
        sigma=2.0,
        min_distance=15,
        threshold_rel=0.2
    )
    
    # Process timelapse
    segmentations, tracks = tracker.process_timelapse(data, method='watershed')
    
    # Compute features for first timepoint
    features = tracker.compute_cell_features(data[0], segmentations[0])
    print(f"Detected {len(features)} cells")
    print(features.head())
    
    # Visualize (requires napari installed)
    # viewer = tracker.visualize_with_napari(data, tracks)
    
    return tracker, segmentations, tracks

In [ ]:
# def segment_with_cellpose(volume: np.ndarray, model_type: str = 'nuclei'):
#     """
#     Use Cellpose for more robust 3D segmentation.
    
#     Requires: pip install cellpose
#     """
#     from cellpose import models
    
#     model = models.Cellpose(model_type=model_type, gpu=True)
    
#     # Run segmentation
#     masks, flows, styles, diams = model.eval(
#         volume,
#         diameter=30,  # Adjust based on your cell size
#         channels=[0, 0],  # Grayscale
#         do_3D=True,
#         anisotropy=2.0  # Z-spacing / XY-spacing
#     )
    
#     return masks

In [ ]:
if __name__ == "__main__":
    # Run example
    tracker, segmentations, tracks = example_usage()
    print(f"Tracking complete: {len(tracks['particle'].unique())} cells tracked")

In [ ]:
data = zarr_file[0:1, 2, :, 100:1000, :]
viewer = tracker.visualize_with_napari(data, tracks)

# Keep the viewer open
napari.run()